# Live Transaction Streaming, SQL Server version

This notebook inserts new transactions directly into your **RetailDashboard** database on SQL Server, every couple of seconds, so your DirectQuery Power BI report updates live.

**Important: do NOT run a backfill here.** Your 30 days of history already exist in SQL Server from the earlier setup. This notebook only adds new live rows on top.

**Before running this:**
1. Make sure `pyodbc` is installed: run `pip install pyodbc` in a terminal if you haven't.
2. Make sure SQL Server is running and you can already connect to RetailDashboard in SSMS.

This uses **Windows Authentication** to connect, since you're running this on the same PC as SQL Server, so no password is stored in this file.

In [2]:
import pyodbc
import random
import time
from datetime import datetime

# I connect using Windows Authentication (Trusted_Connection),
# since this runs on the same machine as SQL Server
SERVER = r"DESKTOP-FSOM5TM\SQLEXPRESS"
DATABASE = "RetailDashboard"

CONN_STRING = (
    f"DRIVER={{ODBC Driver 17 for SQL Server}};"
    f"SERVER={SERVER};"
    f"DATABASE={DATABASE};"
    f"Trusted_Connection=yes;"
)

## Store and product data

This must match what's already in your SQL Server tables, so I'm keeping the same lists.

In [3]:
STORES = [
    (1, "Newmarket", "Auckland", 1.3),
    (2, "Sylvia Park", "Auckland", 1.5),
    (3, "Botany", "Auckland", 1.1),
    (4, "Albany", "Auckland", 1.0),
    (5, "Riccarton", "Christchurch", 0.9),
]

PRODUCTS = [
    (1, "Electric Shaver", "Grooming", 149.00, 3),
    (2, "Beard Trimmer", "Grooming", 79.00, 5),
    (3, "Hair Clipper", "Grooming", 99.00, 4),
    (4, "Hair Dryer", "Hair Care", 129.00, 4),
    (5, "Straightener", "Hair Care", 159.00, 3),
    (6, "Electric Toothbrush", "Oral Care", 119.00, 4),
    (7, "Replacement Blades", "Accessories", 29.00, 9),
    (8, "Shaving Foam", "Accessories", 9.50, 10),
    (9, "Travel Case", "Accessories", 24.00, 5),
    (10, "Gift Set", "Gifts", 89.00, 2),
]

## Connect to SQL Server

In [4]:
conn = pyodbc.connect(CONN_STRING, autocommit=False)
cursor = conn.cursor()
print("Connected to SQL Server successfully")

Connected to SQL Server successfully


## Build one transaction

In [5]:
def make_transaction(outage_store=None):
    """I build one realistic transaction using the current time."""
    weights = [s[3] for s in STORES]
    store = random.choices(STORES, weights=weights)[0]

    # I simulate an outage at one store, so the drop alert has something real to catch
    if outage_store is not None and store[0] == outage_store and random.random() < 0.8:
        return None

    product = random.choices(PRODUCTS, weights=[p[4] for p in PRODUCTS])[0]
    quantity = random.choices([1, 2, 3], weights=[80, 15, 5])[0]
    price = round(product[3] * random.choice([1.0, 1.0, 1.0, 0.9, 0.85]), 2)
    revenue = round(price * quantity, 2)

    return (datetime.now(), store[0], product[0], quantity, price, revenue)

## Stream live transactions into SQL Server

This runs until you stop it. Interrupt the kernel (the stop/square button) to end it.

In [6]:
INSERT_SQL = """
INSERT INTO transactions (transaction_time, store_id, product_id, quantity, unit_price, revenue)
VALUES (?, ?, ?, ?, ?, ?)
"""

def stream(interval=2.0, outage_store=None):
    print(f"Streaming live transactions every ~{interval}s into SQL Server. Interrupt the kernel to stop.")
    count = 0
    try:
        while True:
            tx = make_transaction(outage_store)
            if tx:
                cursor.execute(INSERT_SQL, tx)
                conn.commit()
                count += 1
                if count % 5 == 0:
                    print(f"{count} live transactions inserted")
            time.sleep(random.uniform(0.3, 1.5) * interval)
    except KeyboardInterrupt:
        print(f"Stopped after {count} live transactions")

## Run it

Leave this running while Power BI is open, then click Refresh in Power BI to see new data arrive.

In [ ]:
stream(interval=2.0)

Streaming live transactions every ~2.0s into SQL Server. Interrupt the kernel to stop.
5 live transactions inserted
10 live transactions inserted


In [11]:
stream(interval=2.0, outage_store=3)

Streaming live transactions every ~2.0s into SQL Server. Interrupt the kernel to stop.
5 live transactions inserted
10 live transactions inserted
15 live transactions inserted
20 live transactions inserted
25 live transactions inserted
30 live transactions inserted
35 live transactions inserted
40 live transactions inserted
45 live transactions inserted
50 live transactions inserted
55 live transactions inserted
60 live transactions inserted
65 live transactions inserted
70 live transactions inserted
75 live transactions inserted
80 live transactions inserted
85 live transactions inserted
90 live transactions inserted
95 live transactions inserted
100 live transactions inserted
105 live transactions inserted
110 live transactions inserted
115 live transactions inserted
120 live transactions inserted
125 live transactions inserted
130 live transactions inserted
135 live transactions inserted
140 live transactions inserted
145 live transactions inserted
150 live transactions inserted
155